In [ ]:
from fireworks import Fireworks
import os
from pathlib import Path

PROJECT_ROOT = Path("../")
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(override=True)

import os
FIREWORKS_API_KEY = os.getenv("FIREWORKS_API_KEY")
FIREWORKS_ACCOUNT_ID = os.getenv("FIREWORKS_ACCOUNT_ID")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")
WANDB_ENTITY_NAME = os.getenv("WANDB_ENTITY_NAME")
# %%

In [ ]:
DATASET_ID = "" # <- Fireworks Dataset ID here 
BASE_MODEL = "accounts/fireworks/models/kimi-k2p5"

### SFT Job Creation

In [ ]:
# Kimi-K2.5 model's chat template is not suitable for training, So I use customized chat template

chat_template = r"""{%- macro render_content(msg) -%}
    {%- set c = msg.get('content') -%}
    {%- if c is string -%}
      {{ c }}
    {%- elif c is not none -%}
      {% for content in c -%}
        {% if content['type'] == 'image' or content['type'] == 'image_url' -%}
          <|media_begin|>image<|media_content|><|media_pad|><|media_end|>
        {% elif content['type'] == 'video' or content['type']== 'video_url'-%}
          <|kimi_k25_video_placeholder|>
        {% else -%}
          {{ content['text'] }}
        {%- endif -%}
      {%- endfor -%}
    {%- endif -%}
{%- endmacro -%}
{% macro set_roles(message) -%}
  {%- set role_name =  message.get('name') or  message['role'] -%}
  {%- if message['role'] == 'user' -%}
    <|im_user|>{{role_name}}<|im_middle|>
  {%- elif message['role'] == 'assistant' -%}
    <|im_assistant|>{{role_name}}<|im_middle|>
  {%- else -%}
    <|im_system|>{{role_name}}<|im_middle|>
  {%- endif -%}
{%- endmacro -%}
{%- macro render_toolcalls(message) -%}
  <|tool_calls_section_begin|>
  {%- for tool_call in message['tool_calls'] -%}
    {%- set formatted_id = tool_call['id'] -%}
    <|tool_call_begin|>{{ formatted_id }}<|tool_call_argument_begin|>{% if tool_call['function']['arguments'] is string %}{{ tool_call['function']['arguments'] }}{% else %}{{ tool_call['function']['arguments'] | tojson }}{% endif %}<|tool_call_end|>
  {%- endfor -%}
  <|tool_calls_section_end|>
{%- endmacro -%}
{# Find last non-tool-call assisitant message #}
{%- set ns = namespace(last_non_tool_call_assistant_msg=-1) -%}

{# split all messages into history & suffix, reasoning_content in suffix should be reserved.#}
{%- set hist_msgs = messages[:ns.last_non_tool_call_assistant_msg] -%}
{%- set suffix_msgs = messages[ns.last_non_tool_call_assistant_msg:] -%}
{%- if tools -%}
  {%- if tools_ts_str -%}
    <|im_system|>tool_declare<|im_middle|>{{ tools_ts_str }}<|im_end|>
  {%- else -%}
    <|im_system|>tool_declare<|im_middle|>{{ tools | tojson(separators=(',', ':')) }}<|im_end|>
  {%- endif -%}
{%- endif -%}
  
{%- for message in hist_msgs -%}
  {{set_roles(message)}}
  {%- if message['role'] == 'assistant' -%}
    <think></think>{{render_content(message)}}
    {%- if message.get('tool_calls') -%}
      {{render_toolcalls(message)}}
    {%- endif -%}
  {%- elif message['role'] == 'tool' -%}
    {%- set tool_call_id = message.tool_call_id -%}
    ## Return of {{ tool_call_id }}
{{render_content(message)}}
  {%- elif message['content'] is not none -%}
    {{render_content(message)}}
  {%- endif -%}
  <|im_end|>
{%- endfor -%}
{%- for message in suffix_msgs -%}
  {{set_roles(message)}}
  {%- if message['role'] == 'assistant' -%}
    {%- if thinking is defined and thinking is false -%}
    <think></think>{{render_content(message)}}
    {%- else -%}
    {%- set rc = message.get('reasoning_content', '') -%}
    <think>{{rc}}</think>{{render_content(message)}}
    {%- endif -%}
    {%- if message.get('tool_calls') -%}
     {{render_toolcalls(message)}}
    {%- endif -%}
  {%- elif message['role'] == 'tool' -%}
    {%- set tool_call_id = message.tool_call_id -%}
    ## Return of {{ tool_call_id }}
{{render_content(message)}}
  {%- elif message['content'] is not none -%}
    {{render_content(message)}}
  {%- endif -%}
  <|im_end|>
{%- endfor -%}
{%- if add_generation_prompt -%}
  <|im_assistant|>assistant<|im_middle|>
  {%- if thinking is defined and thinking is false -%}
  <think></think>
  {%- else -%}
  <think>
  {%- endif -%}
{%- endif -%}"""

In [ ]:
client = Fireworks(api_key=FIREWORKS_API_KEY)

wandb_config = {
    "enabled": True,
    "apiKey": WANDB_API_KEY,
    "project": "kimi-k2p5-sft-lora",
    "entity": WANDB_ENTITY_NAME
}

job = client.supervised_fine_tuning_jobs.create(
    account_id=FIREWORKS_ACCOUNT_ID,
    dataset=DATASET_ID,
    base_model=BASE_MODEL,
    max_context_length=20000,
    jinja_template=chat_template,
    lora_rank=16,
    epochs=2,
    batch_size_samples=1,
    gradient_accumulation_steps=32,
    learning_rate=4e-6,
    learning_rate_warmup_steps=100,
    optimizer_weight_decay=0.01,
    wandb_config=wandb_config,
)

print(f"✅ SFT Job Created! ID: {job.name}")